# 16 - Defense Baselines (review #9)

**Jalankan di SageMaker SETELAH `cleaned_100.pkl` + UNSW CSV tersedia.**

Menjawab review #9: membandingkan pertahanan kami (few-shot + FGSM adversarial training)
dengan beberapa strategi pertahanan alternatif yang serius, semuanya di ATAS few-shot
(agar generalisasi lintas-jaringan terjaga) dan diuji pada sumbu ketahanan yang sama
(PCFS FGSM/PGD adaptive).

Varian yang dibandingkan:
- **fewshot** (referensi, tanpa pertahanan evasion).
- **fs+adv_fgsm** (pendekatan kami: FGSM-based adversarial training).
- **fs+adv_pgd** (baseline: PGD-based adversarial training -- yang diminta reviewer/JISA).
- **fs+gauss_aug** (baseline non-adversarial: Gaussian noise augmentation training).
- **fs+rand_smooth** (baseline bersertifikat: randomized smoothing saat inferensi).

5 seed, rerata; util identik nb14 (saliency, PCFS, FGSM/PGD). Angka NYATA -> baris LaTeX tab:defense_baselines.

In [ ]:
import importlib, subprocess, sys
for pkg,imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost'),('scipy','scipy'),('boto3','boto3')]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import os, json, pickle
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, precision_score, recall_score, confusion_matrix
from xgboost import XGBClassifier
from scipy import stats

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
OUTDIR='paper2_reviewer_out'; os.makedirs(OUTDIR,exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); REGION='ap-southeast-1'
SEEDS=[13,42,101,202,303]
H=0.01; EPS_TRAIN=0.1; ADV_RATIO=0.20; FEWSHOT_FRAC=0.01; MAXN=40000
EPS_EVAL=[0.05,0.1,0.2]; PGD_ITERS=10; PGD_ALPHA=0.02
SIGMA_SMOOTH=0.1; N_SMOOTH=20   # randomized smoothing: noise std (z-space) + jumlah sampel
print('files:', os.path.exists(CIC_PKL), os.path.exists(UNSW_TRAIN), os.path.exists(UNSW_TEST))

In [ ]:
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys()); IX={c:i for i,c in enumerate(CANON)}
def build_matrix(df,side):
    idx=0 if side=='cic' else 1; cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON; out=out.replace([np.inf,-np.inf],np.nan)
    return out.fillna(out.median(numeric_only=True)).fillna(0.0).astype(float).values
def make_xgb(seed):
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=seed,tree_method='hist')
def loss_bin(model,X,y):
    p=np.clip(model.predict_proba(X)[:,1],1e-15,1-1e-15); y=y.astype(float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
def saliency(model,X,y,h=H):
    n,m=X.shape; S=np.zeros((n,m))
    for i in range(m):
        Xp=X.copy(); Xp[:,i]+=h; Xm=X.copy(); Xm[:,i]-=h
        S[:,i]=(loss_bin(model,Xp,y)-loss_bin(model,Xm,y))/(2*h)
    return S
def project_functional(Xs_scaled, mean, scale, X_ref_scaled=None):
    Xo=Xs_scaled*scale+mean; Xo=np.clip(Xo,0.0,None)
    Xo[:,IX['fwd_pkts']]=np.round(Xo[:,IX['fwd_pkts']]); Xo[:,IX['bwd_pkts']]=np.round(Xo[:,IX['bwd_pkts']])
    Xo[:,IX['fwd_bytes']]=np.maximum(Xo[:,IX['fwd_bytes']],Xo[:,IX['fwd_pkts']])
    Xo[:,IX['bwd_bytes']]=np.maximum(Xo[:,IX['bwd_bytes']],Xo[:,IX['bwd_pkts']])
    if X_ref_scaled is not None:
        Xr=X_ref_scaled*scale+mean
        for j in [IX[c] for c in ['fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','duration']]:
            Xo[:,j]=np.maximum(Xo[:,j],Xr[:,j])
    with np.errstate(divide='ignore',invalid='ignore'):
        fm=np.where(Xo[:,IX['fwd_pkts']]>0,Xo[:,IX['fwd_bytes']]/Xo[:,IX['fwd_pkts']],0.0)
        bm=np.where(Xo[:,IX['bwd_pkts']]>0,Xo[:,IX['bwd_bytes']]/Xo[:,IX['bwd_pkts']],0.0)
    Xo[:,IX['fwd_mean']]=fm; Xo[:,IX['bwd_mean']]=bm
    with np.errstate(divide='ignore',invalid='ignore'):
        dd=np.where(Xo[:,IX['duration']]>0,Xo[:,IX['duration']],np.nan)
        Xo[:,IX['src_load']]=np.nan_to_num((Xo[:,IX['fwd_bytes']]+Xo[:,IX['bwd_bytes']])/dd,nan=0.0)
        Xo[:,IX['dst_load']]=np.nan_to_num(Xo[:,IX['bwd_pkts']]/dd,nan=0.0)
    return (Xo-mean)/scale
def fgsm_functional(X,S,eps,mean,scale):
    return project_functional(X+eps*np.sign(S),mean,scale,X_ref_scaled=X)
def pgd_functional(model,X,y,eps,mean,scale,iters=PGD_ITERS,alpha=PGD_ALPHA):
    Xadv=X.copy()
    for _ in range(iters):
        S=saliency(model,Xadv,y)
        Xadv=Xadv+alpha*np.sign(S)
        Xadv=np.clip(Xadv,X-eps,X+eps)
        Xadv=project_functional(Xadv,mean,scale,X_ref_scaled=X)
    return Xadv
def cm_metrics(y,yp):
    tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    return dict(mcc=float(matthews_corrcoef(y,yp)),f1=float(f1_score(y,yp,zero_division=0)),
                recall=float(recall_score(y,yp,zero_division=0)),
                precision=float(precision_score(y,yp,zero_division=0)))
print('util siap')

In [ ]:
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float); sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0); y_cic=(np.asarray(d['y'])!=benign).astype(int)
unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values
Xc_all=build_matrix(cic_df,'cic'); Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
print('loaded CIC',Xc_all.shape,'UNSW tr/te',Xu_tr_raw.shape,Xu_te_raw.shape)

## Bangun 5 varian di atas few-shot + evaluasi ketahanan (PCFS FGSM/PGD)

randomized smoothing dievaluasi dgn prediksi ber-noise (majority vote), termasuk saat diserang.

In [ ]:
def adv_augment(base,Xtr,ytr,rng,attack):
    n=min(MAXN,len(Xtr)); idx=rng.choice(len(Xtr),n,replace=False)
    if attack=='fgsm':
        S=saliency(base,Xtr[idx],ytr[idx]); Xa=Xtr[idx]+EPS_TRAIN*np.sign(S)
    else:  # pgd
        Xa=pgd_functional(base,Xtr[idx],ytr[idx],EPS_TRAIN,MEAN,SCALE)
    na=min(int(len(Xtr)*ADV_RATIO/(1-ADV_RATIO)),len(Xa)); sel=rng.choice(len(Xa),na,replace=False)
    return np.vstack([Xtr,Xa[sel]]),np.concatenate([ytr,ytr[idx][sel]])
def gauss_augment(Xtr,ytr,rng,sigma=0.1):
    n=min(MAXN,len(Xtr)); idx=rng.choice(len(Xtr),n,replace=False)
    Xa=Xtr[idx]+rng.normal(0,sigma,Xtr[idx].shape)
    na=min(int(len(Xtr)*ADV_RATIO/(1-ADV_RATIO)),len(Xa)); sel=rng.choice(len(Xa),na,replace=False)
    return np.vstack([Xtr,Xa[sel]]),np.concatenate([ytr,ytr[idx][sel]])
def smooth_predict(model,X,rng,sigma=SIGMA_SMOOTH,n=N_SMOOTH):
    votes=np.zeros(len(X))
    for _ in range(n):
        Xn=X+rng.normal(0,sigma,X.shape)
        votes+=(model.predict_proba(Xn)[:,1]>=0.5).astype(int)
    return (votes/n>=0.5).astype(int)

def build_defenses(direction,Xsrc,ysrc,Xtgt_tr,ytgt_tr,seed):
    global MEAN,SCALE
    rng=np.random.RandomState(seed); M={}
    nfs=max(1,int(len(Xtgt_tr)*FEWSHOT_FRAC)); ifs=rng.choice(len(Xtgt_tr),nfs,replace=False)
    Xfs=np.vstack([Xsrc,Xtgt_tr[ifs]]); yfs=np.concatenate([ysrc,ytgt_tr[ifs]])
    M['fewshot']=make_xgb(seed).fit(Xfs,yfs)
    Xa1,ya1=adv_augment(M['fewshot'],Xfs,yfs,rng,'fgsm'); M['fs_adv_fgsm']=make_xgb(seed).fit(Xa1,ya1)
    Xa2,ya2=adv_augment(M['fewshot'],Xfs,yfs,rng,'pgd');  M['fs_adv_pgd']=make_xgb(seed).fit(Xa2,ya2)
    Xg,yg=gauss_augment(Xfs,yfs,rng);                     M['fs_gauss_aug']=make_xgb(seed).fit(Xg,yg)
    M['fs_rand_smooth']=M['fewshot']   # smoothing = inference-time; model dasar = fewshot
    return M

def eval_defenses(direction,M,mean,scale,Xt,yt,seed,rows):
    rng=np.random.RandomState(seed+777)
    for v,m in M.items():
        smooth=(v=='fs_rand_smooth')
        yp_clean=smooth_predict(m,Xt,rng) if smooth else m.predict(Xt)
        r={'direction':direction,'seed':seed,'defense':v,'condition':'clean'}; r.update(cm_metrics(yt,yp_clean)); rows.append(r)
        S=saliency(m,Xt,yt)
        for e in EPS_EVAL:
            Xf=fgsm_functional(Xt,S,e,mean,scale)
            ypf=smooth_predict(m,Xf,rng) if smooth else m.predict(Xf)
            rf={'direction':direction,'seed':seed,'defense':v,'condition':f'fgsm_eps{e}'}; rf.update(cm_metrics(yt,ypf)); rows.append(rf)
            Xp=pgd_functional(m,Xt,yt,e,mean,scale)
            ypp=smooth_predict(m,Xp,rng) if smooth else m.predict(Xp)
            rp={'direction':direction,'seed':seed,'defense':v,'condition':f'pgd_eps{e}'}; rp.update(cm_metrics(yt,ypp)); rows.append(rp)

ALL=[]
for seed in SEEDS:
    Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=seed,stratify=y_cic)
    scc=StandardScaler().fit(Xc_tr_raw); Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
    scu=StandardScaler().fit(Xu_tr_raw); Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
    MEAN,SCALE=scu.mean_,scu.scale_
    Mc=build_defenses('CIC->UNSW',Xc_tr,yc_tr,Xu_tr,y_utr,seed)
    eval_defenses('CIC->UNSW',Mc,scu.mean_,scu.scale_,Xu_te,y_ute,seed,ALL)
    MEAN,SCALE=scc.mean_,scc.scale_
    Mu=build_defenses('UNSW->CIC',Xu_tr,y_utr,Xc_tr,yc_tr,seed)
    eval_defenses('UNSW->CIC',Mu,scc.mean_,scc.scale_,Xc_te,yc_te,seed,ALL)
    print('seed',seed,'selesai')
dfb=pd.DataFrame(ALL); dfb.to_csv(os.path.join(OUTDIR,'defense_baselines_raw.csv'),index=False)
print('total rows',len(dfb))

In [ ]:
agg=dfb.groupby(['direction','defense','condition']).agg(
    mcc_mean=('mcc','mean'),mcc_std=('mcc','std'),recall_mean=('recall','mean'),precision_mean=('precision','mean')).reset_index()
agg.to_csv(os.path.join(OUTDIR,'defense_baselines_agg.csv'),index=False)
import IPython.display as ipd; ipd.display(agg.round(3))
# --- baris LaTeX tab:defense_baselines: clean + FGSM eps0.1 + PGD eps0.1 per arah ---
DEF=[('fewshot','few-shot (no evasion defense)'),('fs_adv_fgsm','\\textbf{few-shot+adv (FGSM, ours)}'),
     ('fs_adv_pgd','few-shot+adv (PGD-AT)'),('fs_gauss_aug','few-shot+Gaussian-aug'),
     ('fs_rand_smooth','few-shot+rand-smoothing')]
def gm(direction,defense,cond):
    s=agg[(agg['direction']==direction)&(agg['defense']==defense)&(agg['condition']==cond)]
    return float(s['mcc_mean'].iloc[0]) if len(s) else float('nan')
def fmt(v):
    if not np.isfinite(v): return '--'
    return '$'+f'{v:.3f}'.replace('.',',')+'$'
lines=[]
for direction in ['CIC->UNSW','UNSW->CIC']:
    lines.append('\\multirow{5}{*}{'+direction.replace('->','$\\rightarrow$')+'}')
    for dk,dl in DEF:
        cells=[fmt(gm(direction,dk,'clean')),fmt(gm(direction,dk,'fgsm_eps0.1')),fmt(gm(direction,dk,'pgd_eps0.1'))]
        lines.append(f' & {dl} & '+' & '.join(cells)+' \\\\')
    lines.append('\\midrule' if direction=='CIC->UNSW' else '')
latex='\n'.join(l for l in lines if l!='')
open(os.path.join(OUTDIR,'defense_baselines_rows.tex'),'w').write(latex)
print('=== TEMPEL baris berikut ke Tabel tab:defense_baselines ===\n'); print(latex)

In [ ]:
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.startswith('defense_baselines') and fn.endswith(('.csv','.tex')):
            s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'unsw-far/paper2_reviewer/{fn}'); up+=1
    print('upload',up,'-> s3://%s/unsw-far/paper2_reviewer/'%S3_BUCKET)
except Exception as e: print('upload gagal:',e)